# Final Cell Annotation

Binarization-based marker annotation (scBoolSeq) combined with Leiden-cluster majority-vote label propagation. Moved out of `1. Cell Annotation.ipynb` to isolate the working end-to-end flow from exploratory/alternative methods.


In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore") # umap deprecation warnings related to numba do not concern us.

In [2]:
import functools as fn
import itertools as it

import numpy as np
import pandas as pd

import sklearn
from sklearn.utils import Bunch
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.manifold import TSNE
from sklearn.preprocessing import FunctionTransformer
from umap import umap_ as umap

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotnine as pn
from plotnine import *

In [3]:
import scanpy as sc

from scboolseq.utils import parse_data_directory, parse_pickles
from scboolseq import scBoolSeq

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
from pipeline_config import load_config

cfg = load_config()  # dataset wybierasz w pipeline_config.toml (`active`) albo env BOOLAI_DATASET
print(cfg.describe())

## a) Based on binarized states and literature markers

### HVG binarization

Here we import Highy Variable Genes (HVGs) previously identified in `0. Data Preparation.ipynb`.

In [ ]:
data = pd.read_csv(cfg.path("hvgs_csv"), index_col=0)
print(data.shape)
data.head()

In [64]:
%%time
scbool = scBoolSeq(
    dor_threshold=0.995, # To retain one of the markers, otherwise it will be marked as discarded due to its high dropout rate
    confidence=.75 # To maximize binarization
)


CPU times: user 13 μs, sys: 1 μs, total: 14 μs
Wall time: 15.7 μs


In [65]:
# Jeśli wiesz, że obiekt przechowuje model w self.gmm:
if hasattr(scbool, 'gmm'):
    scbool.gmm.max_iter = 10000
    scbool.gmm.tol = 1e-3

In [66]:
scbool.max_iter=1000
scbool.covariance_type='diag'

In [ ]:
import joblib

scbool.fit(data)
joblib.dump(scbool, cfg.path("binarizer_joblib"))

In [83]:
scbool.criteria_.Category.value_counts()

Category
ZeroInf    326
Bimodal    162
Name: count, dtype: int64

Here we import the markers from the file set as `markers_file` in `pipeline_config.toml` (by default `markers.py`: markers identified/mentioned in *Herault, 2021* and *Herault, 2023* papers; other datasets can point to a different file in the same format).  
We filter out genes missing in our hvg set followingly.

In [ ]:
# markers z pliku wskazanego w pipeline_config.toml (`markers_file`) — ten sam kontrakt co `from markers import ...`
_markers = cfg.load_markers_module()
positive_markers = _markers.positive_markers  # function to generate a configuration dictionnary from a set of marker genes
states_and_markers = _markers.states_and_markers  # Dictionnary containing phenotypes and their marker genes
print(f"markers: {cfg.markers_path.name} ({len(states_and_markers)} states)")

marker_genes = pd.Series(list(set(it.chain.from_iterable(states_and_markers.values()))))
marker_genes.shape[0]

In [85]:
marker_genes = marker_genes[marker_genes.isin(data.columns)]

In [86]:
scbool.criteria_.loc[marker_genes, :].Category.value_counts()

Category
Bimodal    25
ZeroInf    22
Name: count, dtype: int64

In [87]:
%time 
bin_data = scbool.binarize(data)

CPU times: user 2 μs, sys: 2 μs, total: 4 μs
Wall time: 13.1 μs


NotFittedError: This GaussianMixtureBinarizer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

Get a partial view of how marker genes were binarized:

In [479]:
partial_bin_configs = bin_data[marker_genes]
partial_bin_configs.fillna('').head()

NameError: name 'bin_data' is not defined

In [480]:
prior_markers = {
    obs: positive_markers(markers)
    for obs, markers
    in states_and_markers.items()
}
prior_markers

{'iHSC': {'Slamf1': 1,
  'Fli1': 1,
  'Srf': 1,
  'Procr': 1,
  'Bclaf1': 1,
  'Foxp1': 1,
  'Kdm5b': 1,
  'Alcam': 1},
 'srHSC': {'Hmmr': 1,
  'Mki67': 1,
  'Ccnb1': 1,
  'Zbtb7a': 1,
  'Srf': 1,
  'Bclaf1': 1},
 'ifnHSC': {'Stat1': 1, 'Irf9': 1, 'Irf7': 1, 'Irf1': 1},
 'qHSC': {'Nr4a1': 1,
  'Gata2': 1,
  'Junb': 1,
  'Fli1': 1,
  'Egr1': 1,
  'Tal1': 1,
  'Cdkn1a': 1},
 'preDiff': {'Cd34': 1, 'Sox4': 1, 'Myc': 1, 'Spi1': 1, 'Hoxa9': 1},
 'pLymph': {'Ly6d': 1,
  'Il7r': 1,
  'Zbtb16': 1,
  'Cd79a': 1,
  'Tcrg-C4': 1,
  'Gata3': 1,
  'Ikzf1': 1,
  'Trp53inp1': 1},
 'pNeuMast': {'Hdc': 1,
  'Mcpt8': 1,
  'Elane': 1,
  'Fcer1a': 1,
  'Spi1': 1,
  'Cebpa': 1},
 'pEr': {'Klf1': 1, 'Kif1': 1, 'Zfpm1': 1, 'Gata1': 1},
 'pMk': {'Zfpm1': 1,
  'Gata1': 1,
  'Pf4': 1,
  'Fli1': 1,
  'Uhrf1': 1,
  'Vwf': 1,
  'Clu': 1}}

We have to filter out markers missing from data from this dictionary as well.

In [ ]:
for state in prior_markers:
    prior_markers[state] = {marker: 1 for marker in prior_markers[state] if marker in data.columns}

In [ ]:
prior_markers

In [ ]:
obs_by_markers = {}
for _conf, _marker_dict in prior_markers.items():
    # Build a query with the genes and their values, configuration by configuration
    _conf_query = " & ".join(
        f"`{_gene}` == {_value:.1f}" for _gene, _value in _marker_dict.items()
    )    
    _matching = partial_bin_configs.fillna(-1.0).query(_conf_query).index
    obs_by_markers.update({_conf: _matching})

In [ ]:
number_of_matching_obs = {conf: idx.shape[0] for conf, idx in obs_by_markers.items()}
number_of_matching_obs

In [ ]:
percentage_matching = {conf: 100*idx.shape[0]/partial_bin_configs.shape[0] for conf, idx in obs_by_markers.items()}
{k: f"{v:.2f}" for k, v in percentage_matching.items()}

In [ ]:
int(sum(percentage_matching.values()))

Only 8% of cells have boolean values matching exactly markers reported in the papers.  
This is not the final result, because we want to explore loosening the conditions for cell annotation based on identified markers.

## Check for multi-labelled cells

In [ ]:
intersections_of_all_combinations = list(map(
    lambda x: (
        x, 
        obs_by_markers[x[0]].intersection(obs_by_markers[x[1]]).shape[0]
    ), 
    it.combinations(obs_by_markers, 2) # All possible pairwise combinations, without replacement
)) # This will contain a list of nested tuples as follows: ((conf_1, conf_2), number_of_cells_in_both_groups)

In [ ]:
non_null_intersections = list(filter(
    lambda p1_p2_shape: p1_p2_shape[-1].shape[0] > 0,
    map(
        lambda x: (x, obs_by_markers[x[0]].intersection(obs_by_markers[x[1]])), 
        it.combinations(obs_by_markers, 2)
    )
))
print(len(non_null_intersections))
len(non_null_intersections) / len(intersections_of_all_combinations)

In [ ]:
{(c1, c2): len(idx) for ((c1, c2), idx) in non_null_intersections}

In [ ]:
unambiguous_observations = {}
for _phenotype, _idx in obs_by_markers.items():
    _unambiguous = fn.reduce(
        lambda x, y: x.difference(y),
        [idx for ((p1, p2), idx) in non_null_intersections if p1 == _phenotype or p2 == _phenotype],
        _idx
    )
    unambiguous_observations.update({_phenotype: _unambiguous})
    _n_unambiguous = _unambiguous.shape[0]
    print(f"{_phenotype} has {_n_unambiguous} matching unambiguous configurations, {100*_n_unambiguous/(number_of_matching_obs[_phenotype]+0.0001):.2f}% of matching configs")
    # added small number to avoid 0 multiplication error - HOTFIX
unlabelled_idx = data.index.difference(fn.reduce(lambda x, y: x.union(y), unambiguous_observations.values()))

Cell states that are presented here are the result of clusterization method described in *Herault, 2023*.  
They are **defined** there as their proposed classification of HSC cell subtypes.

In [ ]:
color_map = {
    "Unknown": "#e6e6e6",   # gray
    "iHSC": "#1f77b4",      # blue
    "srHSC": "#17becf",     # light blue / cyan
    "ifnHSC": "#9467bd",    # purple
    "qHSC": "#2ca02c",      # green
    "preDiff": "#ff7f0e",   # orange
    "pLymph": "#e377c2",    # pink
    "pNeuMast": "#8c564b",  # brown
    "pEr": "#d62728",       # red
    "pMk": "#bcbd22",       # olive
}
# states from a different markers file without a color above get a fallback one
_no_color = [s for s in states_and_markers if s not in color_map]
color_map.update(dict(zip(_no_color, sns.color_palette("tab20", len(_no_color)).as_hex())))

In [ ]:
unambiguous_metadata = pd.DataFrame({
    "label": "Unknown"
}, index=data.index)
for _conf, _idx in unambiguous_observations.items():
    unambiguous_metadata.loc[_idx, "label"] = _conf

color = unambiguous_metadata.label.map(color_map)
color.name = "label_color"
unambiguous_metadata = unambiguous_metadata.join(color)

print(100 * unambiguous_metadata.label.value_counts() / unambiguous_metadata.shape[0])
unambiguous_metadata.head()

In [ ]:
unambiguous_metadata.query("label != 'Unknown'").shape[0] / unambiguous_metadata.shape[0]

In [ ]:
EXPORT_METADATA = True

if EXPORT_METADATA:
    (
        unambiguous_metadata
            .query("label != 'Unknown'")
            .rename(columns={"label": "observation"})
            .observation
            .to_frame()
            .to_csv(cfg.path("scboolseq_obs_csv"))
    )
    #unambiguous_metadata.to_csv("GSE122466_unambiguous_metadata_batch1.csv")

In [ ]:
N_PCS = 400

In [ ]:
%%time
projection_pipeline = Pipeline([
    ('pca', PCA()),
    ('subset_max_pcas', FunctionTransformer(lambda x: x.iloc[:, :N_PCS] if isinstance(x, pd.DataFrame) else x[:, :N_PCS])), 
    ('umap', umap.UMAP())
])
reduced_umap = projection_pipeline.fit_transform(data)
projected_umap = pd.DataFrame(reduced_umap, index=data.index)

In [ ]:
%%time
projection_pipeline2 = Pipeline([
    ('pca', PCA()),
    ('subset_max_pcas', FunctionTransformer(lambda x: x.iloc[:, :N_PCS] if isinstance(x, pd.DataFrame) else x[:, :N_PCS])), 
    ('tsne', TSNE())
])
reduced_tsne = projection_pipeline2.fit_transform(data)
projected_tsne = pd.DataFrame(reduced_tsne, index=data.index)

Here we present cumulative variance explained by top PCs.

In [ ]:
(
    pd.Series(projection_pipeline['pca'].explained_variance_ratio_)
        .cumsum()
        .to_frame()
        .reset_index()
        .rename(columns={'index': 'n_components', 0: 'cummulative_explained_variance'})
        .plot.scatter(x='n_components', y='cummulative_explained_variance')
)
plt.grid()
plt.axvline(N_PCS)

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(16,5))

_tsne_i = 0
_umap_i = 1
_ = axs[_umap_i].scatter(projected_umap.loc[unlabelled_idx, 0], projected_umap.loc[unlabelled_idx, 1], label="Unknown", alpha=.5, c=color_map["Unknown"])
for name, idx in unambiguous_observations.items():
        _x, _y = projected_umap.loc[idx, 0], projected_umap.loc[idx, 1]
        axs[_umap_i].scatter(_x, _y, label=name, alpha=1 if name == 'HC' else .3)#, c=color_map[name])

axs[_umap_i].set_xlabel("UMAP 1")
axs[_umap_i].set_ylabel("UMAP 2")
#axs[0].legend()

_xname = projected_tsne.columns[0]
_yname = projected_tsne.columns[1]
_ = axs[_tsne_i].scatter(projected_tsne.loc[unlabelled_idx, _xname], projected_tsne.loc[unlabelled_idx, _yname], label="Unknown", alpha=.3, c=color_map["Unknown"])
for name, idx in unambiguous_observations.items():
        _x, _y = projected_tsne.loc[idx, _xname], projected_tsne.loc[idx, _yname]
        axs[_tsne_i].scatter(_x, _y, label=name, alpha=1 if name == 'HC' else .3)#, c=color_map[name])
axs[_tsne_i].set_xlabel("tSNE 1")
axs[_tsne_i].set_ylabel("tSNE 2")
#axs[1].legend()
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())

plt.suptitle("Cell labels determined by Boolean marker gene signatures")
fig.tight_layout()

In [ ]:
isolated_cells = (
    projected_umap
        .rename(columns={c: f"umap_{c}" for c in projected_umap.columns})
        .query("umap_0 > umap_0.mean() & umap_1 < 0")
        .index
)

In [ ]:
print(f"{100*isolated_cells.shape[0]/data.shape[0]:.1f}% of cells are in this region/cluster")

In [ ]:
isolated_cells_labels: set = set()
for name, idx in unambiguous_observations.items():
    if isolated_cells.isin(idx).any():
        isolated_cells_labels.update({name})
        
isolated_cells_labels

## Clustering and label propagation

We will now look at unsupervised clustering (Leiden) which is common practice in scRNA-Seq.
We will use a voting scheme to label the unsupervised clusters: Each cluster recieves the label of the majority of cells within (we only consider cells which are actually labelled).
In this part, we use the `scanpy` package for scRNA-Seq analysis.

In [ ]:
from pipeline_config import ensure_corrected_h5ad

# integrate=true: wynik '2. Data Integration'; integrate=false: równoważna macierz bez ComBata zbudowana z merged_dataset.h5ad
adata = sc.read_h5ad(ensure_corrected_h5ad(cfg))
N_PCS = min(N_PCS, adata.n_vars - 1)  # PCA wymaga n_comps < liczba genów
adata

### Wczytanie poprawionej macierzy (batch + cykl komórkowy) zamiast surowego `dataset_hvgs.csv`

Klastrowanie Leiden niżej (na którym oprzemy etykietowanie majority-vote) liczyło się wcześniej wprost na `data` (log-normalizowane HVG z `dataset_hvgs.csv`), bez żadnej korekty. W `scHSC_herault` finalne, publikowane klastry (Fig. 1b) powstają dopiero **po** integracji batchy i `ScaleData(vars.to.regress = "G2M_score+S_score+G1_score")` (`R_src/Seurat3_integration.R:141-161`: integracja → regresja → PCA → UMAP/Leiden) — więc tu też PCA/klastrowanie ma iść na w pełni skorygowanych danych, nie na surowych.

`2. Data Integration.ipynb` produkuje teraz dokładnie taką macierz: `integrated_combat_corrected_expression.h5ad` — ten sam kanoniczny zestaw HVG co `dataset_hvgs.csv` (patrz `docs/scHSC_herault_cell_cycle_i_standaryzacja.md` §6), skorygowany o batch (`scanpy.pp.combat` — nie scanorama, bo `correct_scanpy` po drodze L2-normalizuje każdą komórkę i psuje analogię do `scale.data`, patrz §4) i o cykl komórkowy (`regress_out`+`scale`, ten sam mechanizm co dawniej liczony tutaj lokalnie — teraz przeniesiony do notebooka 2).

Wczytujemy ją tu zamiast budować `adata` z `data`. Binaryzacja scBoolSeq + dopasowanie markerowe w sekcji (a) wyżej zostają bez zmian, na surowym `dataset_hvgs.csv` — majority-vote niżej łączy je z klastrami Leiden po indeksie komórki, nie po zestawie genów, więc to nie przeszkadza.

**Dwuetapowe etykietowanie, nie cykl**: `2. Data Integration.ipynb` korzysta z **wstępnych** etykiet (`preliminary_cell_annotation_metadata.tsv`, transfer learning scVI/scArches z `1. Cell Annotation.ipynb`) do decyzji o `discard` *przed* integracją — nie z wyniku tego notebooka. Ten notebook liczy dopiero **finalne** etykiety (na już zintegrowanych/skorygowanych danych) i eksportuje je pod osobną nazwą, `cell_annotation_metadata.tsv` — te dwa pliki celowo się różnią, więc przepływ jest liniowy (1 → 2 → 3), bez cyklu. Patrz `docs/scHSC_herault_cell_cycle_i_standaryzacja.md` §5.

In [ ]:
sc.pp.pca(adata, n_comps=N_PCS)

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=N_PCS)

In [ ]:
sc.pp.neighbors(adata, n_pcs=N_PCS)
sc.tl.umap(adata)
sc.tl.tsne(adata, n_pcs=N_PCS)

In [ ]:
%%time
sc.tl.leiden(
    adata, key_added="leiden_res0_25", resolution=0.25, flavor="igraph", n_iterations=2
)
sc.tl.leiden(
    adata, key_added="leiden_res0_5", resolution=0.5, flavor="igraph", n_iterations=2
)
sc.tl.leiden(
    adata, key_added="leiden_res1", resolution=1.0, flavor="igraph", n_iterations=2
)

Here we visually compare which clusters were found with different resolutions of Leiden clusterization method.

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_res0_25", "leiden_res0_5", "leiden_res1"],
    legend_loc="on data",
)

In [ ]:
sc.tl.leiden(
    adata, resolution=1.0, flavor="igraph", n_iterations=2
) # This execution is needed just for adata update with simple "leiden" named layer

In [ ]:
100 * adata.obs.leiden.value_counts() / adata.n_obs

In [ ]:
# projected_umap/projected_tsne (sekcja a) to inny embedding na innej populacji
# komórek (14915, surowe 488 HVG) niż adata tutaj (14100, skorygowane 2000 HVG) —
# używamy embeddingów policzonych na tym samym adata, na którym liczyliśmy Leiden.
gg_umap = pd.DataFrame(
    adata.obsm["X_umap"], index=adata.obs_names, columns=["umap_1", "umap_2"]
).join(adata.obs.leiden)
gg_tsne = pd.DataFrame(
    adata.obsm["X_tsne"], index=adata.obs_names, columns=["t-SNE_1", "t-SNE_2"]
).join(adata.obs.leiden)

In [ ]:
_plot_frame = gg_umap
first_plot = (
    ggplot(_plot_frame, aes(x=_plot_frame.columns[0], y=_plot_frame.columns[1]))
        + geom_point(aes(color='leiden'), alpha=.5)
)
first_plot

These 2 cells below look like something that might be not applicable to our data (was only needed specifically for original's paper data).

**Aktualizacja**: ten heurystyczny próg (konkretna ćwiartka UMAP: `umap_1 > mean & umap_2 < 0`) był dobrany wzrokowo dla starego, nieskorygowanego osadzenia na 488 genach. UMAP nie ma z góry ustalonej orientacji (dowolny obrót/odbicie między przebiegami, zwłaszcza na innym zestawie genów i innej korekcie danych), więc ten sam próg na nowym, skorygowanym 2000-genowym embeddingu może trafić w zupełnie inny (albo żaden sensowny) klaster. Zamiast ufać mu automatycznie, `discarded_leiden` niżej jest ustawione na `None` (nic nie jest odrzucane), dopóki ktoś nie zweryfikuje wzrokowo na wykresie UMAP wyżej, czy istnieje analogiczny klaster-artefakt i który to numer.

In [ ]:
# Próg z poprzedniego (nieskorygowanego) embeddingu — patrz uwaga w komórce
# markdown wyżej. Zostawione tylko jako punkt odniesienia, nie ufamy mu ślepo.
_to_discard = _plot_frame.query("umap_1 > umap_1.mean() & umap_2 < 0").leiden.unique()
_to_discard

In [ ]:
# Nie ufamy automatycznie _to_discard[0] (patrz uwaga wyżej) — dopóki ktoś nie
# zweryfikuje wzrokowo, który klaster (jeśli jakikolwiek) faktycznie jest
# artefaktem na nowym embeddingu, nic nie oznaczamy jako discard.
discarded_leiden = None
discarded_leiden

In [ ]:
pheno_by_cluster = {}
for clus, frame in _plot_frame.groupby('leiden'):
    louv_idx = frame.index
    pheno_by_cluster.update({
        clus: {
            pheno: louv_idx.isin(pheno_idx).sum()
            for pheno, pheno_idx
            in unambiguous_observations.items()
        }
    })

pheno_by_cluster

These are the number of observations matching each one of the labels, for every louvain cluster.

In [ ]:
majority_pheno_by_cluster = {}
for clus, votes in pheno_by_cluster.items():
    votes = sorted(votes.items(), key=lambda item: item[1], reverse=True)
    _total = sum(v for _, v in votes)
    relative_votes = [(name, v/_total) for name, v in votes] 
    print(clus, relative_votes)
    majority_pheno_by_cluster.update({clus: relative_votes[0][0]})

Here we selected the majority label for each one of the clusters.

In [ ]:
represented_by_leiden = set(majority_pheno_by_cluster.values())
phenotypes = set(unambiguous_observations)

In [ ]:
100 * len(represented_by_leiden.intersection(phenotypes)) / len(phenotypes)

In [ ]:
phenotypes.difference(represented_by_leiden)

In [ ]:
_new_plot = _plot_frame.join(_plot_frame.leiden.map(majority_pheno_by_cluster).to_frame().rename(columns={"leiden": "phenotype"}))
_new_plot.head()

In [ ]:
second_plot = (
    ggplot(_new_plot, aes(x=_plot_frame.columns[0], y=_plot_frame.columns[1]))
        + geom_point(aes(color='phenotype'), alpha=.5)
)
second_plot

Here we can see the final result of the binarization-based annotation method.

In [ ]:
palette = sns.color_palette('Set2', _plot_frame.leiden.unique().shape[0])
palette.as_hex()

In [ ]:
final_color_map = dict(zip(_new_plot.phenotype.unique(),palette.as_hex()))
final_color_map

In [ ]:
metadata = pd.DataFrame({
    'label': _new_plot.phenotype,
    'label_color': _new_plot.phenotype.map(final_color_map),
    'leiden': _new_plot.leiden,
})
metadata.loc[:, 'discard'] = False
metadata.loc[metadata.leiden == discarded_leiden, 'discard'] = True
metadata.head()

In [ ]:
metadata.discard.value_counts()

## Exporting metadata (**annotated cell types**)

Here we export the obtained data, to use it for integration/trajectory inference.

In [ ]:
metadata.to_csv(cfg.path("cell_annotation_tsv"), sep="\t")

In [ ]:
metadata